# Handling Data Imbalance in Machine Learning
### A Comprehensive Guide to Techniques, Intuition, and Practice

---

## Table of Contents

1. **Understanding Data Imbalance** — What it is, why it matters
2. **Evaluation Metrics** — Why accuracy fails, and what to use instead
3. **Data-Level Techniques**
   - Random Oversampling
   - SMOTE and Variants (Borderline-SMOTE, SVM-SMOTE, ADASYN)
   - Random Undersampling
   - Tomek Links
   - Edited Nearest Neighbours (ENN)
   - Hybrid Methods (SMOTE + Tomek, SMOTE + ENN)
4. **Algorithm-Level Techniques**
   - Cost-Sensitive Learning
   - Class Weight Adjustment
   - Threshold Moving / Calibration
5. **Ensemble Methods** — BalancedBagging, EasyEnsemble, RUSBoost
6. **Advanced Approaches**
   - Focal Loss and Custom Loss Functions
   - Anomaly Detection as Imbalance Handling
   - Data Augmentation
7. **Decision Framework** — When to use what
8. **Summary and References**

---

## 1. Understanding Data Imbalance

### What Is Data Imbalance?

Data imbalance refers to a situation in classification tasks where the number of observations belonging to one class is significantly lower (or higher) than observations belonging to other classes. The class with fewer samples is called the **minority class**, and the class with more samples is the **majority class**.

### Degree of Imbalance

| Ratio (Majority:Minority) | Severity | Example |
|---|---|---|
| 10:1 | Mild | Customer churn prediction |
| 100:1 | Moderate | Fraud detection in banking |
| 1000:1 | Severe | Rare disease diagnosis |
| 10000:1+ | Extreme | Network intrusion detection |

### Why Does It Matter?

Most machine learning algorithms assume roughly balanced class distributions. When faced with imbalanced data:

1. **Accuracy Paradox**: A model predicting only the majority class achieves high accuracy but is useless. For a 99:1 ratio, always predicting "not fraud" gives 99% accuracy.

2. **Gradient Domination**: In gradient-based optimization, the majority class dominates the loss landscape. The model learns to minimize loss by ignoring the minority class.

3. **Decision Boundary Bias**: The decision boundary is pushed towards the minority class, making it harder to correctly classify minority samples.

4. **Insufficient Learning Signal**: With very few minority samples, the model cannot learn robust patterns for that class.

### Industrial Examples of Imbalanced Data

| Domain | Problem | Typical Ratio |
|---|---|---|
| Finance | Credit card fraud detection | 99.8% legitimate, 0.2% fraud |
| Healthcare | Cancer diagnosis from scans | 95-99% healthy, 1-5% malignant |
| Manufacturing | Defect detection on assembly lines | 99.9% normal, 0.1% defective |
| Cybersecurity | Intrusion detection systems | 99.99% normal traffic, 0.01% attacks |
| Oil & Gas | Equipment failure prediction | 98% normal operation, 2% failure |
| E-commerce | Purchase conversion prediction | 97% browse-only, 3% purchase |

In [0]:
# ============================================================
# SETUP: Install required libraries and create sample dataset
# ============================================================
%pip install imbalanced-learn scikit-learn matplotlib seaborn numpy pandas -q

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from collections import Counter

# ============================================================
# Create a synthetic imbalanced dataset
# Simulating: Fraud Detection (1% fraud, 99% legitimate)
# ============================================================
X, y = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=15,
    n_redundant=3,
    n_clusters_per_class=2,
    weights=[0.99, 0.01],  # 99% class 0, 1% class 1
    flip_y=0.01,
    random_state=42
)

print("Class Distribution:")
print(f"  Class 0 (Legitimate): {Counter(y)[0]} samples ({Counter(y)[0]/len(y)*100:.1f}%)")
print(f"  Class 1 (Fraud):      {Counter(y)[1]} samples ({Counter(y)[1]/len(y)*100:.1f}%)")
print(f"  Imbalance Ratio:      {Counter(y)[0]/Counter(y)[1]:.1f}:1")

# Visualize the imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of class distribution
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], [Counter(y)[0], Counter(y)[1]], 
            color=['steelblue', 'crimson'], edgecolor='black')
axes[0].set_title('Class Distribution in Fraud Detection Dataset', fontsize=13)
axes[0].set_ylabel('Number of Samples')
axes[0].set_yscale('log')  # Log scale to see both bars

# 2D scatter plot of first two features
axes[1].scatter(X[y==0, 0], X[y==0, 1], alpha=0.3, s=10, label='Legitimate', c='steelblue')
axes[1].scatter(X[y==1, 0], X[y==1, 1], alpha=0.9, s=50, label='Fraud', c='crimson', marker='x')
axes[1].set_title('Feature Space (First 2 Features)', fontsize=13)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Evaluation Metrics for Imbalanced Data

### The Accuracy Paradox

Consider a fraud detection system with 99.8% legitimate and 0.2% fraudulent transactions. A naive classifier that **always predicts "legitimate"** achieves:

$$\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}} = \frac{9980}{10000} = 99.8\%$$

This sounds impressive but catches **zero frauds** — making it completely useless for the business purpose.

### Metrics That Matter

#### Precision

Of all predicted positives, how many are actually positive?

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Intuition**: When you flag a transaction as fraud, how often are you right?

**Use when**: False positives are expensive (e.g., blocking legitimate customers).

#### Recall (Sensitivity / True Positive Rate)

Of all actual positives, how many did we catch?

$$\text{Recall} = \frac{TP}{TP + FN}$$

**Intuition**: Of all actual frauds, what fraction did we detect?

**Use when**: Missing positives is costly (e.g., undetected cancer).

#### F1-Score

Harmonic mean of precision and recall — balances both concerns:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

#### F-beta Score

Generalization where $$\beta$$ controls the trade-off:

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

- $$\beta = 0.5$$: Weights precision higher (spam detection)
- $$\beta = 2$$: Weights recall higher (disease screening)

#### Area Under the ROC Curve (AUC-ROC)

Measures discrimination ability across all thresholds. Plots $$\text{TPR}$$ vs $$\text{FPR}$$:

$$\text{AUC} = \int_0^1 \text{TPR}(\text{FPR}^{-1}(x)) \, dx$$

**Caveat**: Can be overly optimistic under extreme imbalance because FPR denominator (TN + FP) is very large.

#### Area Under the Precision-Recall Curve (AUC-PR)

More informative than ROC under severe imbalance. Focuses only on positive-class performance.

#### Matthews Correlation Coefficient (MCC)

Balanced metric even for highly imbalanced datasets:

$$\text{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

Ranges from $$-1$$ (perfect misclassification) to $$+1$$ (perfect classification), with $$0$$ being random.

#### Cohen's Kappa

Measures agreement beyond chance:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $$p_o$$ is observed accuracy and $$p_e$$ is expected accuracy by chance.

---

**Industrial Guideline**: In fraud detection, practitioners often prioritize **Recall > 0.95** (catch 95%+ of fraud) while maintaining **Precision > 0.05** (at least 1 in 20 flags is real fraud). The exact trade-off depends on the cost of false positives vs. false negatives.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve
)

# ============================================================
# Demonstrate why accuracy is misleading
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train a simple model WITHOUT handling imbalance
model_naive = LogisticRegression(max_iter=1000, random_state=42)
model_naive.fit(X_train, y_train)
y_pred = model_naive.predict(X_test)
y_prob = model_naive.predict_proba(X_test)[:, 1]

# Compare with a "dummy" classifier that always predicts majority
y_dummy = np.zeros_like(y_test)  # Always predict class 0

print("=" * 60)
print("COMPARISON: Naive Model vs Always-Predict-Majority")
print("=" * 60)
print(f"\n{'Metric':<25} {'Naive Model':<15} {'Always Majority':<15}")
print("-" * 55)
print(f"{'Accuracy':<25} {accuracy_score(y_test, y_pred):<15.4f} {accuracy_score(y_test, y_dummy):<15.4f}")
print(f"{'Precision (class 1)':<25} {precision_score(y_test, y_pred, zero_division=0):<15.4f} {precision_score(y_test, y_dummy, zero_division=0):<15.4f}")
print(f"{'Recall (class 1)':<25} {recall_score(y_test, y_pred):<15.4f} {recall_score(y_test, y_dummy):<15.4f}")
print(f"{'F1-Score (class 1)':<25} {f1_score(y_test, y_pred):<15.4f} {f1_score(y_test, y_dummy):<15.4f}")
print(f"{'AUC-ROC':<25} {roc_auc_score(y_test, y_prob):<15.4f} {'N/A':<15}")
print(f"{'AUC-PR':<25} {average_precision_score(y_test, y_prob):<15.4f} {'N/A':<15}")
print(f"{'MCC':<25} {matthews_corrcoef(y_test, y_pred):<15.4f} {matthews_corrcoef(y_test, y_dummy):<15.4f}")

print("\n" + "=" * 60)
print("KEY INSIGHT: Accuracy is nearly identical, but MCC reveals")
print("the dummy classifier is worthless (MCC=0).")
print("=" * 60)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix (Naive Logistic Regression)')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, 'b-', label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Precision-Recall Curve
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(recall_vals, precision_vals, 'r-', label=f'AUC-PR = {average_precision_score(y_test, y_prob):.3f}')
axes[2].axhline(y=Counter(y_test)[1]/len(y_test), color='k', linestyle='--', alpha=0.5, label='Baseline')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Data-Level Techniques: Resampling

### Taxonomy

Data-level techniques modify the training dataset to achieve a more balanced distribution **before** model training. They do not change the learning algorithm itself.

```
Data-Level Techniques
├── Oversampling (increase minority)
│   ├── Random Oversampling
│   ├── SMOTE
│   ├── Borderline-SMOTE
│   ├── SVM-SMOTE
│   └── ADASYN
├── Undersampling (decrease majority)
│   ├── Random Undersampling
│   ├── Tomek Links
│   ├── Edited Nearest Neighbours (ENN)
│   └── Condensed Nearest Neighbours (CNN)
└── Hybrid (combine over + under)
    ├── SMOTE + Tomek Links
    └── SMOTE + ENN
```

---

### 3.1 Random Oversampling

#### Intuition

The simplest approach: **duplicate** random samples from the minority class until the desired ratio is achieved. Like photocopying the rare examples so the model sees them more often.

#### How It Works

1. Identify all samples belonging to the minority class
2. Randomly select samples from the minority class (with replacement)
3. Add copies to the training set until the desired balance is reached

#### Pros and Cons

| Pros | Cons |
|---|---|
| Simple and fast | Risk of overfitting (exact duplicates) |
| No information loss | Increases training time (larger dataset) |
| Works with any classifier | No new information generated |

#### Industrial Example

**Predictive Maintenance at a Wind Farm**: Out of 50,000 hourly sensor readings, only 200 correspond to pre-failure states. Random oversampling duplicates these 200 readings to give the model more exposure to failure patterns. However, since we're using exact copies, the model may memorize specific sensor combinations rather than learning generalizable failure signatures.

#### When to Use

- As a quick baseline before trying more sophisticated methods
- When the minority class is small but representative
- When overfitting risk is low (e.g., with regularized models or ensembles)

In [0]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import f1_score, recall_score

# ============================================================
# Random Oversampling
# ============================================================
print("RANDOM OVERSAMPLING")
print("=" * 50)

# Before oversampling
print(f"\nBefore: {Counter(y_train)}")

# Apply random oversampling
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)

print(f"After:  {Counter(y_ros)}")
print(f"Dataset grew from {len(y_train)} to {len(y_ros)} samples")

# Train and evaluate
model_ros = LogisticRegression(max_iter=1000, random_state=42)
model_ros.fit(X_ros, y_ros)
y_pred_ros = model_ros.predict(X_test)

print(f"\nResults on Test Set:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_ros):.4f} (was {f1_score(y_test, y_pred):.4f})")
print(f"  Recall:    {recall_score(y_test, y_pred_ros):.4f} (was {recall_score(y_test, y_pred):.4f})")
print(f"  Precision: {precision_score(y_test, y_pred_ros):.4f} (was {precision_score(y_test, y_pred):.4f})")

### 3.2 SMOTE (Synthetic Minority Over-sampling Technique)

#### Intuition

Instead of duplicating existing samples, SMOTE **generates new synthetic samples** by interpolating between existing minority class instances. Think of it as "filling in the gaps" between real minority samples in feature space.

#### Algorithm

1. For each minority sample $$x_i$$, find its $$k$$ nearest neighbours (typically $$k=5$$) from the minority class
2. Randomly select one neighbour $$x_{nn}$$
3. Generate a synthetic sample along the line segment between $$x_i$$ and $$x_{nn}$$:

$$x_{\text{new}} = x_i + \lambda \cdot (x_{nn} - x_i)$$

where $$\lambda \sim \text{Uniform}(0, 1)$$

4. Repeat until the desired number of synthetic samples is created

#### Geometric Interpretation

SMOTE creates new points **on the line segment** connecting two minority samples in the feature space. This expands the decision region of the minority class without merely duplicating existing points.

#### Mathematical Detail

Given a minority sample $$x_i \in \mathbb{R}^d$$ and its neighbour $$x_{nn} \in \mathbb{R}^d$$:

$$x_{\text{new}}^{(j)} = x_i^{(j)} + \lambda \cdot (x_{nn}^{(j)} - x_i^{(j)}), \quad j = 1, 2, \ldots, d$$

This is equivalent to a **convex combination** when $$\lambda \in [0,1]$$:

$$x_{\text{new}} = (1 - \lambda) \cdot x_i + \lambda \cdot x_{nn}$$

#### Pros and Cons

| Pros | Cons |
|---|---|
| Creates novel samples (reduces overfitting vs. duplication) | Can generate noisy samples between classes |
| Expands the minority decision region | Assumes linear interpolation is valid |
| Works well with KNN-based and SVM models | Struggles in high dimensions |
| Widely used industry standard | Doesn't consider majority class distribution |

#### Industrial Example

**Medical Imaging — Rare Disease Detection**: In a dataset of 100,000 retinal scans, only 500 show signs of diabetic macular edema. SMOTE generates synthetic feature vectors by interpolating between similar positive cases. This helps the model learn a broader representation of the disease's presentation without simply memorizing the 500 known cases.

#### Key Hyperparameters

- $$k$$: Number of nearest neighbours (default 5). Lower values create more local synthetic samples.
- `sampling_strategy`: Target ratio after resampling (e.g., 0.5 means minority becomes 50% of majority).

In [0]:
from imblearn.over_sampling import SMOTE

# ============================================================
# SMOTE (Synthetic Minority Over-sampling Technique)
# ============================================================
print("SMOTE - Synthetic Minority Over-sampling Technique")
print("=" * 50)

# Apply SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_smote, y_smote = smote.fit_resample(X_train, y_train)

print(f"Before: {Counter(y_train)}")
print(f"After:  {Counter(y_smote)}")

# Train and evaluate
model_smote = LogisticRegression(max_iter=1000, random_state=42)
model_smote.fit(X_smote, y_smote)
y_pred_smote = model_smote.predict(X_test)

print(f"\nResults on Test Set:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_smote):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_smote):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_smote):.4f}")

# ============================================================
# Visualize SMOTE synthetic generation (2D)
# ============================================================
from imblearn.over_sampling import SMOTE as SMOTE_viz

# Use only 2 features for visualization
X_2d = X_train[:, :2]
smote_viz = SMOTE_viz(k_neighbors=5, random_state=42)
X_2d_res, y_2d_res = smote_viz.fit_resample(X_2d, y_train)

# Identify original vs synthetic samples
n_original = len(X_2d)
X_synthetic = X_2d_res[n_original:]
y_synthetic = y_2d_res[n_original:]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before SMOTE
axes[0].scatter(X_2d[y_train==0, 0], X_2d[y_train==0, 1], alpha=0.3, s=10, c='steelblue', label='Majority')
axes[0].scatter(X_2d[y_train==1, 0], X_2d[y_train==1, 1], alpha=0.9, s=50, c='crimson', marker='x', label='Minority')
axes[0].set_title('Before SMOTE', fontsize=13)
axes[0].legend()

# After SMOTE
axes[1].scatter(X_2d[y_train==0, 0], X_2d[y_train==0, 1], alpha=0.3, s=10, c='steelblue', label='Majority')
axes[1].scatter(X_2d[y_train==1, 0], X_2d[y_train==1, 1], alpha=0.9, s=50, c='crimson', marker='x', label='Original Minority')
axes[1].scatter(X_synthetic[:, 0], X_synthetic[:, 1], alpha=0.6, s=30, c='orange', marker='o', label='Synthetic (SMOTE)')
axes[1].set_title('After SMOTE (Synthetic Samples in Orange)', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 SMOTE Variants

#### Borderline-SMOTE

**Problem Solved**: Standard SMOTE generates synthetic samples uniformly across the minority class. But samples deep inside the minority region don't help much — the model already classifies them correctly. The decision boundary is what matters.

**Intuition**: Only generate synthetic samples from minority instances that are near the decision boundary ("borderline" samples). A minority sample is "borderline" if roughly half its $$k$$-neighbours belong to the majority class.

**Algorithm**:
1. For each minority sample $$x_i$$, find $$k$$ nearest neighbours from the **entire** dataset
2. Let $$m$$ = number of majority-class neighbours
3. If $$k/2 \leq m < k$$: $$x_i$$ is a **borderline** (DANGER) sample
4. Apply SMOTE only to borderline samples

**Two sub-types**:
- **Borderline-SMOTE1**: Generates synthetics only between borderline minority samples and their minority neighbours
- **Borderline-SMOTE2**: Can also interpolate toward majority-class neighbours (more aggressive)

---

#### SVM-SMOTE

**Problem Solved**: Identifying the critical boundary region more precisely using an SVM's support vectors.

**Intuition**: Train an SVM first. The support vectors near the boundary are the most informative minority samples. Generate synthetic samples near these support vectors.

**Algorithm**:
1. Train SVM on the data
2. Identify minority-class support vectors
3. Apply SMOTE-like interpolation near these support vectors
4. Extrapolate slightly beyond the minority region for support vectors surrounded by majority samples

---

#### ADASYN (Adaptive Synthetic Sampling)

**Problem Solved**: SMOTE generates the same number of synthetics per minority sample. But harder-to-learn minority samples (those with more majority neighbours) need more synthetic support.

**Intuition**: Generate **more** synthetic samples for minority instances that are harder to classify (surrounded by more majority samples). Adaptively shift the decision boundary toward difficult examples.

**Algorithm**:

1. Calculate the density distribution $$\hat{r}_i$$ for each minority sample:

$$\hat{r}_i = \frac{\Delta_i / k}{\sum_{j=1}^{m_s} \Delta_j / k}$$

where $$\Delta_i$$ = number of majority-class samples among the $$k$$-nearest neighbours of minority sample $$i$$, and $$m_s$$ = total minority samples.

2. The number of synthetic samples to generate for $$x_i$$ is:

$$g_i = \hat{r}_i \times G$$

where $$G$$ = total number of synthetic samples needed.

3. Samples near more majority neighbours get more synthetics.

**Industrial Example**: In a **manufacturing defect classification** system for semiconductor wafers, certain types of defects occur in regions that overlap with normal variation. ADASYN focuses synthetic generation on these ambiguous boundary defects, helping the model distinguish subtle defects from normal process variation.

In [0]:
from imblearn.over_sampling import BorderlineSMOTE, SVMSMOTE, ADASYN

# ============================================================
# Compare SMOTE Variants
# ============================================================
print("COMPARISON OF SMOTE VARIANTS")
print("=" * 60)

variants = {
    'Standard SMOTE': SMOTE(random_state=42),
    'Borderline-SMOTE': BorderlineSMOTE(random_state=42, kind='borderline-1'),
    'SVM-SMOTE': SVMSMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42)
}

results = {}

for name, sampler in variants.items():
    try:
        X_res, y_res = sampler.fit_resample(X_train, y_train)
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_res, y_res)
        y_pred_var = model.predict(X_test)
        
        results[name] = {
            'F1': f1_score(y_test, y_pred_var),
            'Recall': recall_score(y_test, y_pred_var),
            'Precision': precision_score(y_test, y_pred_var),
            'Samples': len(y_res)
        }
    except Exception as e:
        results[name] = {'F1': 0, 'Recall': 0, 'Precision': 0, 'Samples': 0}
        print(f"  {name} failed: {e}")

# Display results
print(f"\n{'Variant':<20} {'F1':<8} {'Recall':<8} {'Precision':<10} {'Total Samples':<15}")
print("-" * 60)
for name, metrics in results.items():
    print(f"{name:<20} {metrics['F1']:<8.4f} {metrics['Recall']:<8.4f} {metrics['Precision']:<10.4f} {metrics['Samples']:<15}")

print("\nNote: Results vary by dataset. Borderline/SVM-SMOTE often excel")
print("when the boundary between classes is complex.")

### 3.4 Random Undersampling

#### Intuition

The mirror of oversampling: instead of adding minority samples, **remove** random majority samples until balance is achieved. Like discarding excess data to level the playing field.

#### How It Works

1. Identify all samples belonging to the majority class
2. Randomly remove majority samples (without replacement) until the desired ratio is reached
3. Train on the reduced dataset

#### The Information Loss Trade-off

If you have 9,900 majority and 100 minority samples and undersample to 100:100, you **discard 9,800 majority samples** — that's 98% of your data! The model may lose important patterns about the majority class.

However, the trade-off is:
- Much **faster** training (smaller dataset)
- Forces the model to focus on discriminative features rather than memorizing the majority class
- Often works surprisingly well with ensemble methods (see Section 5)

#### Pros and Cons

| Pros | Cons |
|---|---|
| Reduces training time significantly | Loses potentially important majority information |
| Can improve model generalization | Model may underfit majority class |
| Simple to implement | High variance in results (depends on which samples removed) |
| Effective with ensembles | Not suitable when majority class is already small |

#### Industrial Example

**Anti-Money Laundering (AML)**: A bank processes 10 million daily transactions, of which ~1,000 are suspicious. Random undersampling creates manageable training sets of 1,000 suspicious + 1,000 legitimate transactions. While 9,999,000 legitimate transactions are discarded, the bank mitigates this by: (a) running multiple rounds with different subsets, and (b) ensuring the sampled legitimate transactions are representative of different transaction types.

#### When to Use

- When the dataset is very large and training time matters
- When combined with ensemble methods (BalancedBagging, EasyEnsemble)
- When majority class has high redundancy
- As a complement to oversampling (hybrid approaches)

In [0]:
from imblearn.under_sampling import RandomUnderSampler

# ============================================================
# Random Undersampling
# ============================================================
print("RANDOM UNDERSAMPLING")
print("=" * 50)

print(f"\nBefore: {Counter(y_train)}")

# Apply random undersampling
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)

print(f"After:  {Counter(y_rus)}")
print(f"Dataset shrank from {len(y_train)} to {len(y_rus)} samples")
print(f"Information lost: {(1 - len(y_rus)/len(y_train))*100:.1f}% of training data discarded")

# Train and evaluate
model_rus = LogisticRegression(max_iter=1000, random_state=42)
model_rus.fit(X_rus, y_rus)
y_pred_rus = model_rus.predict(X_test)

print(f"\nResults on Test Set:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_rus):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_rus):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_rus):.4f}")
print(f"\nTrade-off: Very high recall but lower precision (more false positives)")

### 3.5 Tomek Links

#### Intuition

Instead of randomly removing majority samples, Tomek Links identifies **specific** majority samples that are "too close" to minority samples and removes them. This **cleans the decision boundary** by removing ambiguous, borderline majority samples.

#### Definition

A pair of samples $$(x_i, x_j)$$ forms a **Tomek Link** if:

1. $$x_i$$ belongs to the majority class and $$x_j$$ belongs to the minority class
2. $$x_i$$ is the nearest neighbour of $$x_j$$ AND $$x_j$$ is the nearest neighbour of $$x_i$$
3. They are each other's closest sample from a different class

Formally, $$(x_i, x_j)$$ is a Tomek Link if there exists no sample $$x_k$$ such that:

$$d(x_i, x_k) < d(x_i, x_j) \quad \text{or} \quad d(x_j, x_k) < d(x_i, x_j)$$

where $$d(\cdot, \cdot)$$ is the Euclidean distance.

#### What Happens After Identification

- **Undersampling mode**: Remove the majority-class member of each Tomek Link pair
- **Cleaning mode**: Remove both members of the pair (removes noise from both classes)

#### Geometric Interpretation

Tomek Links are pairs of samples from opposite classes that are each other's nearest neighbours. These pairs lie on the **decision boundary**. By removing the majority member, we:
- Sharpen the boundary between classes
- Remove potentially noisy/mislabelled majority samples
- Give the minority class more "breathing room"

#### Pros and Cons

| Pros | Cons |
|---|---|
| Removes noise at the boundary | Only removes a few samples (mild effect) |
| Doesn't randomly discard useful data | Computationally expensive (nearest neighbour search) |
| Improves decision boundary clarity | Insufficient alone for severe imbalance |
| Good as a cleaning step after SMOTE | May not help if classes are well-separated |

#### Industrial Example

**Credit Scoring**: In credit default prediction, some customers are very close to the default/non-default boundary (e.g., they have 1-2 late payments but haven't technically defaulted). These ambiguous cases form Tomek Links. Removing the "safe" member of these pairs clarifies what the boundary between default and non-default truly looks like, helping the model be more decisive in edge cases.

#### When to Use

- As a **cleaning step** after oversampling (SMOTE + Tomek Links)
- When you want targeted removal rather than random undersampling
- When the decision boundary is noisy

In [0]:
from imblearn.under_sampling import TomekLinks

# ============================================================
# Tomek Links
# ============================================================
print("TOMEK LINKS")
print("=" * 50)

print(f"\nBefore: {Counter(y_train)}")

# Apply Tomek Links
tomek = TomekLinks()
X_tomek, y_tomek = tomek.fit_resample(X_train, y_train)

print(f"After:  {Counter(y_tomek)}")
print(f"Removed {len(y_train) - len(y_tomek)} majority samples (Tomek Link pairs)")
print(f"Note: Only a small number of samples are removed — boundary cleaning only")

# Train and evaluate
model_tomek = LogisticRegression(max_iter=1000, random_state=42)
model_tomek.fit(X_tomek, y_tomek)
y_pred_tomek = model_tomek.predict(X_test)

print(f"\nResults on Test Set:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_tomek):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_tomek):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_tomek):.4f}")
print(f"\nInsight: Tomek Links alone has a mild effect. It's best used")
print(f"as a cleaning step combined with SMOTE.")

### 3.6 Edited Nearest Neighbours (ENN)

#### Intuition

A more aggressive cleaning technique than Tomek Links. ENN removes any majority sample whose class label **disagrees with the majority vote of its $$k$$ nearest neighbours**. This removes noisy and borderline majority samples.

#### Algorithm (Wilson's Editing)

1. For each sample $$x_i$$ in the dataset (or just the majority class):
   - Find its $$k$$ nearest neighbours
   - If the majority vote of those $$k$$ neighbours disagrees with $$x_i$$'s label:
     - Remove $$x_i$$

2. The decision rule:

$$\text{Remove } x_i \text{ if } \sum_{j \in \text{kNN}(x_i)} \mathbb{1}[y_j \neq y_i] > \frac{k}{2}$$

#### Variants

- **ENN (mode)**: Remove if class differs from the **mode** of neighbours' classes
- **All-KNN**: Repeat ENN with increasing values of $$k$$ (from 1 to some max). More aggressive.
- **Repeated ENN (RENN)**: Apply ENN iteratively until no more samples are removed

#### Comparison with Tomek Links

| Aspect | Tomek Links | ENN |
|---|---|---|
| Samples removed | Very few (mutual nearest neighbours only) | More (any misclassified by KNN) |
| Aggressiveness | Mild | Moderate to aggressive |
| Best for | Boundary sharpening | Noise removal |
| Computation | $$O(n^2)$$ for nearest neighbour | $$O(n \cdot k)$$ per sample |

#### Industrial Example

**Insurance Claim Fraud**: In auto insurance, some legitimate claims look very similar to fraudulent ones (e.g., claims filed just before policy expiration). ENN removes these ambiguous legitimate claims that are surrounded by fraudulent claim neighbours, cleaning the decision space so the model can focus on clearer fraud patterns.

#### When to Use

- When the dataset has significant noise or mislabelled samples
- As a cleaning step after SMOTE (SMOTE + ENN)
- When you want more aggressive cleaning than Tomek Links
- When $$k$$ can be tuned on a validation set

In [0]:
from imblearn.under_sampling import EditedNearestNeighbours

# ============================================================
# Edited Nearest Neighbours (ENN)
# ============================================================
print("EDITED NEAREST NEIGHBOURS (ENN)")
print("=" * 50)

print(f"\nBefore: {Counter(y_train)}")

# Apply ENN
enn = EditedNearestNeighbours(n_neighbors=3)
X_enn, y_enn = enn.fit_resample(X_train, y_train)

print(f"After:  {Counter(y_enn)}")
print(f"Removed {len(y_train) - len(y_enn)} samples")
print(f"More aggressive than Tomek Links — removes noisy boundary samples")

# Train and evaluate
model_enn = LogisticRegression(max_iter=1000, random_state=42)
model_enn.fit(X_enn, y_enn)
y_pred_enn = model_enn.predict(X_test)

print(f"\nResults on Test Set:")
print(f"  F1-Score:  {f1_score(y_test, y_pred_enn):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_enn):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_enn):.4f}")

### 3.7 Hybrid Methods: SMOTE + Tomek Links / SMOTE + ENN

#### Intuition

Oversampling alone (SMOTE) can create synthetic samples that fall into majority-class territory — introducing noise. Undersampling alone (Tomek/ENN) doesn't address the fundamental scarcity of minority samples.

**Hybrid methods combine both**: first **oversample** the minority class (SMOTE), then **clean** the resulting dataset by removing noisy samples at the boundary.

#### SMOTE + Tomek Links

1. Apply SMOTE to generate synthetic minority samples
2. Identify Tomek Links in the combined (oversampled) dataset
3. Remove the majority-class member of each Tomek Link pair

**Effect**: The SMOTE step expands the minority region; the Tomek step sharpens the boundary.

#### SMOTE + ENN

1. Apply SMOTE to generate synthetic minority samples
2. Apply ENN to the combined dataset
3. Remove samples (from both classes) that are misclassified by their neighbours

**Effect**: More aggressive cleaning than SMOTE+Tomek. May remove some synthetic minority samples that landed in noisy regions, but the remaining synthetics are higher quality.

#### Comparison

| Method | Cleaning Strength | Risk | Best For |
|---|---|---|---|
| SMOTE alone | None | Noisy synthetics | Quick improvement |
| SMOTE + Tomek | Mild | Minimal loss | Boundary sharpening |
| SMOTE + ENN | Aggressive | May remove useful samples | Noisy datasets |

#### Industrial Example

**Telecom Customer Churn**: A telecom company has 500,000 customers with a 5% churn rate. They apply SMOTE to synthesize additional churn profiles, but some synthetic profiles inadvertently resemble loyal customers who briefly considered switching (borderline cases). SMOTE + ENN removes these ambiguous synthetic profiles, leaving only synthetic churn examples that clearly belong in the churn region of feature space.

#### When to Use

- Default go-to method for moderate-to-severe imbalance
- When SMOTE alone produces too many false positives
- When the boundary between classes is noisy or overlapping

In [0]:
from imblearn.combine import SMOTETomek, SMOTEENN

# ============================================================
# Hybrid Methods: SMOTE + Tomek Links and SMOTE + ENN
# ============================================================
print("HYBRID METHODS")
print("=" * 60)

# --- SMOTE + Tomek Links ---
print("\n--- SMOTE + Tomek Links ---")
smote_tomek = SMOTETomek(random_state=42)
X_st, y_st = smote_tomek.fit_resample(X_train, y_train)
print(f"Before: {Counter(y_train)}")
print(f"After:  {Counter(y_st)}")

model_st = LogisticRegression(max_iter=1000, random_state=42)
model_st.fit(X_st, y_st)
y_pred_st = model_st.predict(X_test)

print(f"F1: {f1_score(y_test, y_pred_st):.4f} | Recall: {recall_score(y_test, y_pred_st):.4f} | Precision: {precision_score(y_test, y_pred_st):.4f}")

# --- SMOTE + ENN ---
print("\n--- SMOTE + ENN ---")
smote_enn = SMOTEENN(random_state=42)
X_se, y_se = smote_enn.fit_resample(X_train, y_train)
print(f"Before: {Counter(y_train)}")
print(f"After:  {Counter(y_se)}")

model_se = LogisticRegression(max_iter=1000, random_state=42)
model_se.fit(X_se, y_se)
y_pred_se = model_se.predict(X_test)

print(f"F1: {f1_score(y_test, y_pred_se):.4f} | Recall: {recall_score(y_test, y_pred_se):.4f} | Precision: {precision_score(y_test, y_pred_se):.4f}")

# --- Summary comparison ---
print("\n" + "=" * 60)
print("SUMMARY: Data-Level Techniques Comparison")
print("=" * 60)
print(f"{'Method':<25} {'F1':<8} {'Recall':<8} {'Precision':<10}")
print("-" * 50)
all_results = {
    'No resampling': y_pred,
    'Random Oversample': y_pred_ros,
    'SMOTE': y_pred_smote,
    'Random Undersample': y_pred_rus,
    'Tomek Links': y_pred_tomek,
    'ENN': y_pred_enn,
    'SMOTE + Tomek': y_pred_st,
    'SMOTE + ENN': y_pred_se
}
for name, preds in all_results.items():
    print(f"{name:<25} {f1_score(y_test, preds):<8.4f} {recall_score(y_test, preds):<8.4f} {precision_score(y_test, preds):<10.4f}")

## 4. Algorithm-Level Techniques

Algorithm-level techniques modify the learning process itself rather than the data. They make the model "pay more attention" to the minority class during training.

---

### 4.1 Cost-Sensitive Learning

#### Intuition

Assign different **misclassification costs** to different classes. Misclassifying a minority sample (fraud missed) should be penalized more heavily than misclassifying a majority sample (legitimate flagged as fraud).

The learning algorithm minimizes **expected cost** rather than simple error rate.

#### Mathematical Framework

Define a **cost matrix** $$C$$ where $$C(i, j)$$ is the cost of predicting class $$j$$ when the true class is $$i$$:

$$C = \begin{bmatrix} C(0,0) & C(0,1) \\ C(1,0) & C(1,1) \end{bmatrix} = \begin{bmatrix} 0 & C_{FP} \\ C_{FN} & 0 \end{bmatrix}$$

For imbalanced problems, typically $$C_{FN} \gg C_{FP}$$ (missing fraud costs much more than a false alarm).

#### How It Modifies the Loss

Standard loss (e.g., cross-entropy):

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

Cost-sensitive loss:

$$\mathcal{L}_{\text{cost}} = -\frac{1}{N} \sum_{i=1}^{N} \left[ w_1 \cdot y_i \log(\hat{p}_i) + w_0 \cdot (1 - y_i) \log(1 - \hat{p}_i) \right]$$

where $$w_1 = C_{FN}$$ and $$w_0 = C_{FP}$$. The weights tilt the loss landscape so that minority-class errors contribute more.

#### The Expected Cost Minimization

The optimal prediction for sample $$x$$ minimizes expected cost:

$$\hat{y} = \arg\min_{j} \sum_{i} P(Y=i|X=x) \cdot C(i, j)$$

For binary classification, predict class 1 if:

$$P(Y=1|X=x) \cdot C_{FN} > P(Y=0|X=x) \cdot C_{FP}$$

This effectively **shifts the decision threshold** from 0.5 to:

$$t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$$

#### Industrial Example

**Healthcare — Cancer Screening**: Missing a malignant tumour (false negative) may result in patient death — a cost in millions and immeasurable human suffering. Flagging a benign growth as suspicious (false positive) leads to an additional biopsy costing ~$1,000 and patient anxiety. A cost matrix might assign:
- $$C_{FN} = 1,000,000$$ (missed cancer)
- $$C_{FP} = 1,000$$ (unnecessary biopsy)

This 1000:1 cost ratio dramatically shifts the decision boundary toward catching every possible cancer.

#### When to Use

- When business costs of different errors are quantifiable
- When you cannot or should not modify the training data
- Works natively with most scikit-learn classifiers via `class_weight`
- Especially effective with tree-based methods and SVMs

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# Cost-Sensitive Learning with Custom Cost Matrix
# ============================================================
print("COST-SENSITIVE LEARNING")
print("=" * 60)

# Define cost ratios (how much more costly is a false negative?)
cost_ratios = [1, 10, 50, 100, 500]

print(f"\n{'Cost Ratio (FN:FP)':<22} {'F1':<8} {'Recall':<8} {'Precision':<10} {'FP':<6} {'FN':<6}")
print("-" * 60)

for ratio in cost_ratios:
    # class_weight maps class labels to weights
    # Higher weight on class 1 = penalize FN more
    model_cs = LogisticRegression(
        max_iter=1000, 
        random_state=42,
        class_weight={0: 1, 1: ratio}
    )
    model_cs.fit(X_train, y_train)
    y_pred_cs = model_cs.predict(X_test)
    
    cm = confusion_matrix(y_test, y_pred_cs)
    fp = cm[0, 1]
    fn = cm[1, 0]
    
    print(f"{ratio:<22} {f1_score(y_test, y_pred_cs):<8.4f} "
          f"{recall_score(y_test, y_pred_cs):<8.4f} "
          f"{precision_score(y_test, y_pred_cs):<10.4f} "
          f"{fp:<6} {fn:<6}")

print("\nObservation: As cost ratio increases, recall rises (fewer FN)")
print("but precision drops (more FP). The optimal ratio depends on")
print("the actual business cost of each error type.")

### 4.2 Class Weight Adjustment

#### Intuition

A special case of cost-sensitive learning where the weights are derived **automatically** from the class distribution. The `class_weight='balanced'` parameter in scikit-learn implements this.

#### The Formula

For a dataset with $$n$$ samples, $$K$$ classes, and $$n_k$$ samples in class $$k$$:

$$w_k = \frac{n}{K \cdot n_k}$$

This ensures that $$\sum_k w_k \cdot n_k = n$$, so each class contributes equally to the total weighted loss.

**Example**: For 9,900 majority and 100 minority samples:
- $$w_0 = \frac{10000}{2 \times 9900} = 0.505$$
- $$w_1 = \frac{10000}{2 \times 100} = 50.0$$

The minority class gets ~100x the weight of the majority class — proportional to the imbalance ratio.

#### Effect on Gradient Updates

For gradient-based methods (logistic regression, neural networks), the weight multiplies the gradient contribution of each sample:

$$\nabla \mathcal{L}_{\text{weighted}} = \sum_{i=1}^{N} w_{y_i} \cdot \nabla \ell(\hat{y}_i, y_i)$$

Minority samples now have gradients $$50\times$$ larger, so the model adjusts its parameters proportionally more to correctly classify them.

#### For Tree-Based Methods

In decision trees with weighted samples, the Gini impurity becomes:

$$\text{Gini}_{\text{weighted}} = 1 - \sum_{k=1}^{K} \left( \frac{\sum_{i: y_i=k} w_k}{\sum_{i} w_{y_i}} \right)^2$$

Splits are chosen to maximize weighted information gain, favouring splits that separate minority samples.

#### Supported Algorithms

| Algorithm | Parameter | Notes |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Adjusts intercept + coefficients |
| SVM | `class_weight='balanced'` | Adjusts margin penalty |
| Random Forest | `class_weight='balanced'` | Per-tree weighting |
| XGBoost | `scale_pos_weight` | Ratio of neg/pos |
| LightGBM | `is_unbalance=True` | Auto-calculates weights |
| Neural Networks | Custom loss weights | Via loss function |

#### Industrial Example

**E-commerce Conversion Prediction**: An online retailer wants to predict which visitors will purchase (3% conversion rate). Using `class_weight='balanced'` with a Random Forest, the model automatically learns that correctly identifying a purchaser is ~33x more important than correctly identifying a browser. This helps the marketing team target the right users for promotional emails without manually tuning weights.

#### When to Use

- **First thing to try** — zero data modification, just one parameter change
- When you want a quick, principled improvement without resampling
- When the imbalance ratio directly reflects the importance ratio
- Works with most scikit-learn estimators

In [0]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# ============================================================
# Class Weight = 'balanced' across different algorithms
# ============================================================
print("CLASS WEIGHT ADJUSTMENT (class_weight='balanced')")
print("=" * 60)

models = {
    'LogReg (no weight)': LogisticRegression(max_iter=1000, random_state=42),
    'LogReg (balanced)': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'RF (no weight)': RandomForestClassifier(n_estimators=100, random_state=42),
    'RF (balanced)': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'RF (balanced_subsample)': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample'),
}

print(f"\n{'Model':<28} {'F1':<8} {'Recall':<8} {'Precision':<10} {'AUC-ROC':<8}")
print("-" * 62)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_m = model.predict(X_test)
    y_prob_m = model.predict_proba(X_test)[:, 1]
    
    print(f"{name:<28} {f1_score(y_test, y_pred_m):<8.4f} "
          f"{recall_score(y_test, y_pred_m):<8.4f} "
          f"{precision_score(y_test, y_pred_m):<10.4f} "
          f"{roc_auc_score(y_test, y_prob_m):<8.4f}")

print("\n'balanced' = weight inversely proportional to class frequency")
print("'balanced_subsample' = recomputed per bootstrap sample (RF only)")

### 4.3 Threshold Moving (Decision Threshold Tuning)

#### Intuition

Most classifiers output a **probability** $$P(Y=1|X)$$ and then apply a default threshold of 0.5:

$$\hat{y} = \begin{cases} 1 & \text{if } P(Y=1|X) \geq 0.5 \\ 0 & \text{otherwise} \end{cases}$$

But **0.5 is arbitrary** and optimal only for balanced datasets with equal misclassification costs. For imbalanced problems, a lower threshold catches more minority samples.

#### The Core Idea

Instead of changing the data or the algorithm, we change **how we interpret the model's output**:

$$\hat{y} = \begin{cases} 1 & \text{if } P(Y=1|X) \geq t \\ 0 & \text{otherwise} \end{cases}$$

where $$t \in (0, 1)$$ is chosen to optimize a target metric.

#### How to Choose the Optimal Threshold

**Method 1: Maximize F1-Score**

Search over thresholds and pick the one maximizing the F1-score on validation data:

$$t^* = \arg\max_t F_1(t)$$

**Method 2: Cost-based threshold**

If costs are known, the optimal threshold is:

$$t^* = \frac{C_{FP}}{C_{FP} + C_{FN}}$$

For fraud detection where $$C_{FN} = 100$$ and $$C_{FP} = 1$$:
$$t^* = \frac{1}{1 + 100} \approx 0.01$$

**Method 3: Youden's J-statistic (ROC-based)**

Find the threshold that maximizes:

$$J = \text{Sensitivity} + \text{Specificity} - 1 = \text{TPR} - \text{FPR}$$

This is the point on the ROC curve farthest from the diagonal.

**Method 4: Geometric Mean**

Maximize $$G\text{-Mean} = \sqrt{\text{Sensitivity} \times \text{Specificity}}$$

#### Why This Works

The model has **already learned** a good ranking of samples by probability. The default threshold just doesn't cut the ranking at the right place for imbalanced data. Threshold tuning exploits the full probability distribution without retraining.

#### Important Caveat: Calibration

Threshold tuning works best when probabilities are **well-calibrated** (i.e., when the model says 0.3, roughly 30% of those are actually positive). If probabilities are poorly calibrated, consider applying **Platt scaling** or **isotonic regression** before threshold tuning.

#### Industrial Example

**Spam Filtering at an Email Provider**: A spam classifier produces probability scores. The default 0.5 threshold lets many spam emails through. By analysing the precision-recall trade-off on held-out data, engineers find that a threshold of 0.15 catches 98% of spam while only misclassifying 0.5% of legitimate emails — an acceptable trade-off given that users can check their spam folder.

#### When to Use

- Always, as a post-processing step after any model training
- When you need fine-grained control over the precision-recall trade-off
- When the deployment cost structure is well-defined
- When the model outputs well-calibrated probabilities

In [0]:
from sklearn.metrics import f1_score, precision_recall_curve

# ============================================================
# Threshold Moving / Optimization
# ============================================================
print("THRESHOLD MOVING")
print("=" * 60)

# Use probabilities from the naive model trained earlier
# (without any resampling or class weights)
y_prob_test = model_naive.predict_proba(X_test)[:, 1]

# Method 1: Search for threshold that maximizes F1
thresholds = np.arange(0.01, 0.99, 0.01)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_prob_test >= t).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_t, zero_division=0))

best_threshold_f1 = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

print(f"Default threshold (0.5):")
print(f"  F1={f1_score(y_test, (y_prob_test >= 0.5).astype(int)):.4f}, "
      f"Recall={recall_score(y_test, (y_prob_test >= 0.5).astype(int)):.4f}")

print(f"\nOptimal threshold (max F1): t* = {best_threshold_f1:.2f}")
y_pred_optimal = (y_prob_test >= best_threshold_f1).astype(int)
print(f"  F1={f1_score(y_test, y_pred_optimal):.4f}, "
      f"Recall={recall_score(y_test, y_pred_optimal):.4f}, "
      f"Precision={precision_score(y_test, y_pred_optimal):.4f}")

# Method 2: Youden's J-statistic
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob_test)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold_j = roc_thresholds[best_idx]

print(f"\nYouden's J threshold: t* = {best_threshold_j:.4f}")
y_pred_j = (y_prob_test >= best_threshold_j).astype(int)
print(f"  F1={f1_score(y_test, y_pred_j):.4f}, "
      f"Recall={recall_score(y_test, y_pred_j):.4f}, "
      f"Precision={precision_score(y_test, y_pred_j):.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 vs Threshold
axes[0].plot(thresholds, f1_scores, 'b-', linewidth=2)
axes[0].axvline(x=0.5, color='red', linestyle='--', label='Default (0.5)')
axes[0].axvline(x=best_threshold_f1, color='green', linestyle='--', label=f'Optimal ({best_threshold_f1:.2f})')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('F1-Score')
axes[0].set_title('F1-Score vs Decision Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall trade-off
recalls = [recall_score(y_test, (y_prob_test >= t).astype(int), zero_division=0) for t in thresholds]
precisions = [precision_score(y_test, (y_prob_test >= t).astype(int), zero_division=0) for t in thresholds]

axes[1].plot(thresholds, recalls, 'b-', label='Recall', linewidth=2)
axes[1].plot(thresholds, precisions, 'r-', label='Precision', linewidth=2)
axes[1].axvline(x=best_threshold_f1, color='green', linestyle='--', label=f'Optimal Threshold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision-Recall Trade-off vs Threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Ensemble Methods for Imbalanced Data

#### Intuition

Ensemble methods combine multiple models to improve performance. For imbalanced data, specialized ensembles incorporate resampling **within** the ensemble framework, gaining the benefits of resampling while reducing variance through aggregation.

---

### 5.1 BalancedBaggingClassifier

**Core Idea**: Standard Bagging draws bootstrap samples from the full (imbalanced) dataset. Balanced Bagging **undersamples the majority class** within each bootstrap iteration.

**Algorithm**:
1. For each base estimator $$h_t$$, $$t = 1, \ldots, T$$:
   - Draw all minority samples (or a bootstrap of them)
   - Randomly undersample majority class to match
   - Train $$h_t$$ on this balanced subset
2. Final prediction: majority vote (or average probability) across all $$T$$ estimators

**Why it works**: Each base model sees a balanced dataset (overcoming bias), and aggregation across many random undersamples recovers information about the majority class that any single undersample misses.

---

### 5.2 EasyEnsemble

**Core Idea**: Create multiple **independent** balanced subsets by undersampling, train a separate learner on each, then combine their predictions.

**Algorithm**:
1. Create $$T$$ subsets of the majority class: $$S_1, S_2, \ldots, S_T$$ (each of size $$|\text{minority}|$$)
2. For each subset, combine with all minority samples: $$D_t = S_t \cup \text{Minority}$$
3. Train AdaBoost on each $$D_t$$
4. Combine all AdaBoost models via majority vote

**Mathematical insight**: By drawing $$T$$ independent subsets, the expected coverage of majority-class samples approaches 100% as $$T$$ grows. With $$T = 10$$ subsets from a 99:1 ratio, approximately $$1 - (1 - 1/99)^{10} \approx 10\%$$ of majority samples appear in at least one subset.

---

### 5.3 Balanced Random Forest

**Core Idea**: At each tree's construction, balance the bootstrap sample by undersampling majority class.

**Algorithm**: Same as Random Forest, but each tree is trained on:
- A bootstrap sample of all minority class points
- An equal-sized random sample of majority class points

---

### 5.4 RUSBoost (Random Under-Sampling + Boosting)

**Core Idea**: Integrates random undersampling within the AdaBoost framework.

**Algorithm** (iteration $$t$$):
1. Undersample majority class from the current weighted distribution
2. Train weak learner $$h_t$$ on the balanced subset
3. Compute weighted error: $$\epsilon_t = \sum_{i: h_t(x_i) \neq y_i} D_t(i)$$
4. Update sample weights: $$D_{t+1}(i) = D_t(i) \cdot \exp(\alpha_t \cdot \mathbb{1}[h_t(x_i) \neq y_i])$$
5. Normalize weights

where $$\alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right)$$

---

### 5.5 SMOTEBoost

**Core Idea**: Integrates SMOTE within AdaBoost. Instead of undersampling, **oversample** the minority class at each boosting iteration using SMOTE with the current weight distribution.

---

### Comparison Table

| Method | Resampling | Base Learner | Key Advantage |
|---|---|---|---|
| BalancedBagging | Undersample per bag | Any | Simple, effective |
| EasyEnsemble | Multiple undersamples | AdaBoost | Good for severe imbalance |
| Balanced RF | Undersample per tree | Decision Tree | Fast, scalable |
| RUSBoost | Undersample per round | Weak learner | Boosting + balance |
| SMOTEBoost | SMOTE per round | Weak learner | No information loss |

#### Industrial Example

**Network Intrusion Detection System (NIDS)**: A corporate network generates 100 million packet logs daily, with ~50 being actual intrusion attempts (0.00005% positive rate). EasyEnsemble creates 20 independent balanced subsets from the 100M logs, each containing 50 intrusions + 50 randomly-sampled normal packets. Each subset trains an AdaBoost model. The ensemble of 20 boosted models covers a wide variety of normal traffic patterns while maintaining sensitivity to the rare intrusions.

#### When to Use

- **BalancedBagging**: General purpose, first ensemble to try
- **EasyEnsemble**: Extreme imbalance (>1000:1)
- **Balanced RF**: When you need speed and interpretability
- **RUSBoost**: When boosting methods suit your problem (sequential data, tabular)

In [0]:
from imblearn.ensemble import (
    BalancedBaggingClassifier, 
    BalancedRandomForestClassifier,
    EasyEnsembleClassifier,
    RUSBoostClassifier
)
from sklearn.tree import DecisionTreeClassifier

# ============================================================
# Ensemble Methods for Imbalanced Data
# ============================================================
print("ENSEMBLE METHODS FOR IMBALANCED DATA")
print("=" * 60)

ensemble_models = {
    'Balanced Bagging': BalancedBaggingClassifier(
        n_estimators=50, random_state=42
    ),
    'Balanced Random Forest': BalancedRandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    'EasyEnsemble': EasyEnsembleClassifier(
        n_estimators=10, random_state=42
    ),
    'RUSBoost': RUSBoostClassifier(
        n_estimators=50, random_state=42
    ),
}

# Add standard RF for comparison
from sklearn.ensemble import RandomForestClassifier
ensemble_models['Standard RF (baseline)'] = RandomForestClassifier(
    n_estimators=100, random_state=42
)

print(f"\n{'Model':<28} {'F1':<8} {'Recall':<8} {'Precision':<10} {'AUC-ROC':<8}")
print("-" * 62)

for name, model in ensemble_models.items():
    model.fit(X_train, y_train)
    y_pred_e = model.predict(X_test)
    y_prob_e = model.predict_proba(X_test)[:, 1]
    
    print(f"{name:<28} {f1_score(y_test, y_pred_e):<8.4f} "
          f"{recall_score(y_test, y_pred_e):<8.4f} "
          f"{precision_score(y_test, y_pred_e):<10.4f} "
          f"{roc_auc_score(y_test, y_prob_e):<8.4f}")

print("\nKey insight: Ensemble methods generally provide the best")
print("precision-recall balance without requiring separate resampling steps.")

## 6. Advanced Approaches

### 6.1 Focal Loss and Custom Loss Functions

#### Intuition

Standard cross-entropy loss treats all samples equally. Even "easy" majority samples (classified with high confidence) contribute to the loss. **Focal Loss** down-weights the contribution of easy-to-classify samples, focusing the model's learning effort on hard, misclassified examples — which in imbalanced settings are often minority samples.

#### Standard Cross-Entropy (Binary)

$$\text{CE}(p_t) = -\log(p_t)$$

where $$p_t = p$$ if $$y=1$$ else $$1-p$$ (probability assigned to the true class).

#### Focal Loss (Lin et al., 2017)

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Two key components:

1. **Modulating factor** $$(1 - p_t)^\gamma$$:
   - When $$p_t \to 1$$ (easy, correctly classified): $$(1 - p_t)^\gamma \to 0$$, so loss $$\to 0$$
   - When $$p_t \to 0$$ (hard, misclassified): $$(1 - p_t)^\gamma \to 1$$, so full loss is applied
   - $$\gamma$$ (focusing parameter): Controls how much easy samples are down-weighted
     - $$\gamma = 0$$: Equivalent to standard cross-entropy
     - $$\gamma = 1$$: Moderate focusing
     - $$\gamma = 2$$: Strong focusing (common default)
     - $$\gamma = 5$$: Very aggressive focusing

2. **Class-balancing factor** $$\alpha_t$$:
   - $$\alpha$$ for the minority class (typically set to the inverse frequency)
   - $$(1-\alpha)$$ for the majority class

#### How Focal Loss Addresses Imbalance

| Sample Type | $$p_t$$ | $$(1-p_t)^2$$ | Contribution |
|---|---|---|---|
| Easy majority (correct, confident) | 0.95 | 0.0025 | Negligible |
| Easy majority (correct, moderate) | 0.80 | 0.04 | Small |
| Hard minority (misclassified) | 0.20 | 0.64 | Large |
| Hard minority (very wrong) | 0.05 | 0.9025 | Dominant |

The model focuses almost entirely on the hard cases, which are disproportionately minority samples in imbalanced datasets.

#### Comparison with Weighted Cross-Entropy

| Aspect | Weighted CE | Focal Loss |
|---|---|---|
| Mechanism | Static weight per class | Dynamic weight per sample |
| Handles | Class imbalance | Class imbalance + easy/hard imbalance |
| Information used | Class label only | Class label + model confidence |
| Over-weighting risk | Yes (all minority equally weighted) | No (easy minority down-weighted too) |

#### Industrial Example

**Autonomous Driving — Pedestrian Detection**: In object detection for self-driving cars, >99.9% of candidate image regions are background (negative). Focal Loss was originally designed for this exact scenario (RetinaNet, 2017). The focusing parameter ensures the model doesn't waste capacity on the overwhelming number of easy background patches but instead focuses on challenging cases: partially occluded pedestrians, unusual poses, or low-contrast conditions.

#### When to Use

- Deep learning models (CNNs, transformers) where you control the loss function
- Object detection (where background dominates)
- When both class imbalance AND difficulty imbalance exist
- When simple class weighting leads to too many false positives

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# Focal Loss Implementation (PyTorch)
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss for binary classification.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha (float): Weighting factor for the minority class [0,1]
        gamma (float): Focusing parameter. gamma=0 is standard CE.
        reduction (str): 'mean', 'sum', or 'none'
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Raw logits (before sigmoid), shape (N,)
            targets: Binary labels, shape (N,)
        """
        # Apply sigmoid to get probabilities
        p = torch.sigmoid(inputs)
        
        # p_t: probability assigned to the true class
        p_t = p * targets + (1 - p) * (1 - targets)
        
        # Alpha weighting
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Focal modulating factor
        focal_weight = (1 - p_t) ** self.gamma
        
        # Binary cross-entropy (element-wise)
        bce = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        
        # Final focal loss
        loss = alpha_t * focal_weight * bce
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


# ============================================================
# Demonstration: Effect of gamma on loss landscape
# ============================================================
print("FOCAL LOSS - Effect of Gamma on Loss Values")
print("=" * 60)

# Simulated probabilities for correctly classified sample
p_t_values = np.linspace(0.01, 0.99, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss vs p_t for different gamma values
gammas = [0, 0.5, 1, 2, 5]
for gamma in gammas:
    focal_loss_values = -((1 - p_t_values) ** gamma) * np.log(p_t_values)
    axes[0].plot(p_t_values, focal_loss_values, label=f'γ = {gamma}', linewidth=2)

axes[0].set_xlabel('Probability of True Class (p_t)', fontsize=11)
axes[0].set_ylabel('Focal Loss', fontsize=11)
axes[0].set_title('Focal Loss vs p_t for Different γ Values', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 5)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)

# Plot relative weight (how much each sample contributes)
for gamma in gammas:
    # Relative to CE (gamma=0)
    relative_weight = (1 - p_t_values) ** gamma
    axes[1].plot(p_t_values, relative_weight, label=f'γ = {gamma}', linewidth=2)

axes[1].set_xlabel('Probability of True Class (p_t)', fontsize=11)
axes[1].set_ylabel('Relative Weight (1 - p_t)^γ', fontsize=11)
axes[1].set_title('Modulating Factor: How Much Each Sample Contributes', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].annotate('Easy samples\n(high confidence)', xy=(0.9, 0.1), fontsize=9, ha='center')
axes[1].annotate('Hard samples\n(low confidence)', xy=(0.15, 0.85), fontsize=9, ha='center')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - γ=0: Standard cross-entropy (all samples weighted equally)")
print("  - γ=2: A well-classified sample (p_t=0.9) has its loss reduced by 100x")
print("  - γ=5: Even more aggressive focusing on hard examples")
print("  - This means the model concentrates learning on the minority class")
print("    (which tends to have low p_t) without explicitly resampling.")

### 6.2 Anomaly Detection as Imbalance Handling

#### Intuition

When the imbalance is extreme (>1000:1), the minority class can be treated not as a "class" but as an **anomaly** — a deviation from normal behavior. Instead of learning to distinguish two classes, we learn what "normal" looks like and flag anything that deviates.

#### Paradigm Shift

| Traditional Classification | Anomaly Detection |
|---|---|
| Learns both classes | Learns only the majority (normal) class |
| Needs labelled positives for training | Can train on unlabelled normal data |
| Struggles with extreme imbalance | Designed for it |
| Binary decision boundary | Outlier score / distance |

#### Key Algorithms

**1. Isolation Forest**

Intuition: Anomalies are few and different. In a random tree, anomalies are isolated in fewer splits (shorter path length).

Anomaly score for sample $$x$$:

$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

where $$E[h(x)]$$ is the average path length over all trees, and $$c(n)$$ is the average path length in an unsuccessful binary search tree with $$n$$ elements (normalisation factor).

- $$s \to 1$$: Anomaly (short paths)
- $$s \to 0.5$$: Normal (average paths)
- $$s \to 0$$: Very dense point

**2. One-Class SVM**

Learns a boundary around the normal data in kernel space. Solves:

$$\min_{w, \rho, \xi} \frac{1}{2} \|w\|^2 + \frac{1}{\nu n} \sum_{i=1}^{n} \xi_i - \rho$$

subject to: $$w \cdot \phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0$$

where $$\nu$$ controls the fraction of support vectors / outliers.

**3. Local Outlier Factor (LOF)**

Compares the local density of a point to its neighbours. A point with much lower density than its neighbours is an outlier:

$$\text{LOF}_k(x) = \frac{\sum_{o \in N_k(x)} \frac{\text{lrd}_k(o)}{\text{lrd}_k(x)}}{|N_k(x)|}$$

where $$\text{lrd}_k(x)$$ is the local reachability density.

- LOF $$\approx 1$$: Similar density to neighbours (normal)
- LOF $$\gg 1$$: Much lower density (anomaly)

**4. Autoencoders (Deep Learning)**

Train a neural network to compress and reconstruct normal data. Anomalies have high **reconstruction error** because the network never learned their patterns:

$$\text{Anomaly Score}(x) = \|x - \text{Decoder}(\text{Encoder}(x))\|^2$$

#### Industrial Example

**Semiconductor Manufacturing — Wafer Defect Detection**: A fab produces 10,000 wafers daily with a 0.01% defect rate (1 defective wafer). Training a traditional classifier with 1 positive and 9,999 negatives is impractical. Instead, an autoencoder learns the distribution of normal wafer sensor readings. Any wafer whose reconstruction error exceeds a threshold (e.g., 3 standard deviations above mean) is flagged for human inspection.

#### When to Use

- Extreme imbalance (>1000:1)
- Very few or no labelled minority samples available
- The minority class is genuinely "anomalous" (not just rare but normal)
- Real-time streaming detection scenarios
- When the concept of "normal" is well-defined but "abnormal" has many forms

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score

# ============================================================
# Anomaly Detection Approaches
# ============================================================
print("ANOMALY DETECTION FOR EXTREME IMBALANCE")
print("=" * 60)

# For anomaly detection, we train ONLY on the majority class
# and detect minority class as anomalies
X_train_normal = X_train[y_train == 0]  # Train only on normal/majority
print(f"Training on {len(X_train_normal)} normal samples only")
print(f"Testing on {len(X_test)} samples ({Counter(y_test)[1]} anomalies)")

# Expected contamination rate
contamination = Counter(y_train)[1] / len(y_train)

anomalies_models = {
    'Isolation Forest': IsolationForest(
        n_estimators=100, 
        contamination=contamination,
        random_state=42
    ),
    'One-Class SVM': OneClassSVM(
        kernel='rbf', 
        gamma='scale',
        nu=contamination  # Upper bound on fraction of outliers
    ),
}

print(f"\n{'Model':<22} {'F1':<8} {'Recall':<8} {'Precision':<10}")
print("-" * 48)

for name, model in anomalies_models.items():
    # Fit on normal data only
    model.fit(X_train_normal)
    
    # Predict: returns 1 for inliers, -1 for outliers
    y_pred_anom = model.predict(X_test)
    
    # Convert: -1 (outlier/anomaly) -> 1 (fraud), 1 (inlier) -> 0 (normal)
    y_pred_converted = (y_pred_anom == -1).astype(int)
    
    print(f"{name:<22} {f1_score(y_test, y_pred_converted):<8.4f} "
          f"{recall_score(y_test, y_pred_converted):<8.4f} "
          f"{precision_score(y_test, y_pred_converted, zero_division=0):<10.4f}")

# LOF (only for novelty detection mode)
lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination, novelty=True)
lof.fit(X_train_normal)
y_pred_lof = lof.predict(X_test)
y_pred_lof_converted = (y_pred_lof == -1).astype(int)

print(f"{'Local Outlier Factor':<22} {f1_score(y_test, y_pred_lof_converted):<8.4f} "
      f"{recall_score(y_test, y_pred_lof_converted):<8.4f} "
      f"{precision_score(y_test, y_pred_lof_converted, zero_division=0):<10.4f}")

print("\nNote: Anomaly detection is most effective for EXTREME imbalance")
print("(1000:1+). For moderate imbalance, classification with resampling")
print("typically outperforms anomaly detection.")

### 6.3 Data Augmentation for Imbalanced Data

#### Intuition

For domains with structured transformations (images, text, time-series), generate new minority samples through **domain-specific transformations** that preserve the label. Unlike SMOTE which interpolates in feature space, data augmentation operates in the original data space using semantics-preserving operations.

#### Domain-Specific Augmentation

**Computer Vision (Images)**:
- Rotation, flipping, cropping
- Color jittering, brightness adjustment
- Elastic deformation
- Mixup: $$x_{\text{new}} = \lambda x_i + (1-\lambda) x_j$$, $$y_{\text{new}} = \lambda y_i + (1-\lambda) y_j$$
- CutMix: Paste a region from one image onto another
- GANs (Generative Adversarial Networks): Generate entirely new realistic minority samples

**Natural Language Processing (Text)**:
- Synonym replacement
- Random insertion/deletion/swap
- Back-translation (English → French → English)
- Contextual augmentation using language models
- Paraphrasing with GPT-style models

**Time Series / Signal Data**:
- Time warping
- Window slicing
- Jittering (adding small noise)
- Magnitude warping
- Permutation (shuffling subsequences)

**Tabular Data**:
- SMOTE and variants (covered above)
- Variational Autoencoders (VAE)
- Conditional GANs (CTGAN)
- Gaussian Copula synthesis

#### GAN-Based Augmentation

A Conditional GAN (CGAN) can be trained to generate minority-class samples:

1. Train Generator $$G(z|y=1)$$ to produce samples conditioned on minority class
2. Train Discriminator $$D(x|y=1)$$ to distinguish real from generated minority samples
3. Minimax objective:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x|y)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z|y)|y))]$$

4. After training, sample from $$G$$ to augment the minority class

#### Industrial Example

**Medical Imaging — Skin Lesion Classification**: A dermatology AI system has 100,000 images of benign moles but only 500 images of melanoma. Data augmentation creates 50,000 additional melanoma images through:
- Geometric transforms (rotation, scaling)
- Color-space manipulation (simulating different lighting/skin tones)
- StyleGAN-generated synthetic lesions

This dramatically improves the model's ability to detect melanoma across diverse patient populations and imaging conditions.

#### When to Use

- Unstructured data (images, text, audio, time series)
- When you have domain knowledge about label-preserving transforms
- When the minority class is genuinely underrepresented (not just undersampled)
- When GAN-quality synthetic data is feasible for your domain

## 7. Decision Framework: When to Use What

### Quick Decision Tree

```
Start
  │
  ├─ How severe is the imbalance?
  │   ├─ Mild (10:1) → Try class_weight='balanced' first
  │   ├─ Moderate (100:1) → SMOTE + ENN or Balanced Ensembles
  │   ├─ Severe (1000:1) → EasyEnsemble or Anomaly Detection
  │   └─ Extreme (10000:1+) → Anomaly Detection or Focal Loss
  │
  ├─ What's your data type?
  │   ├─ Tabular → SMOTE variants + Ensembles
  │   ├─ Images → Data Augmentation + Focal Loss
  │   ├─ Text → Augmentation + Class Weights
  │   └─ Time Series → SMOTE (on features) + Window augmentation
  │
  ├─ What's your model?
  │   ├─ Tree-based (RF, XGB) → class_weight or Balanced RF
  │   ├─ Linear (LogReg, SVM) → SMOTE + class_weight
  │   ├─ Neural Network → Focal Loss + class weights
  │   └─ KNN → SMOTE (expands minority region)
  │
  └─ What are your constraints?
      ├─ Need fast training → Undersampling + Ensemble
      ├─ Can't modify data → Cost-sensitive + Threshold tuning
      ├─ Need interpretability → Class weights + single model
      └─ Production deployment → Threshold tuning (most flexible)
```

### Best Practices Checklist

1. **Always start simple**: `class_weight='balanced'` + threshold tuning
2. **Evaluate correctly**: Use F1, AUC-PR, MCC — never accuracy alone
3. **Resample only training data**: Never apply SMOTE/undersampling to test/validation sets
4. **Use stratified splits**: `StratifiedKFold` preserves class ratios in cross-validation
5. **Combine techniques**: SMOTE + ENN + Balanced RF + threshold tuning is often best
6. **Consider the cost structure**: If costs are known, cost-sensitive > resampling
7. **Monitor in production**: Class distributions drift — retrain when imbalance ratio changes

### Common Pitfalls

| Pitfall | Why It's Wrong | Fix |
|---|---|---|
| SMOTE before train/test split | Data leakage (synthetic test samples informed by training) | Always split first, resample only training set |
| Using accuracy as metric | Masks poor minority performance | Use F1, AUC-PR, MCC |
| Over-resampling | Creates perfect balance when mild imbalance exists | Use `sampling_strategy=0.5` instead of 1.0 |
| Ignoring calibration | Threshold tuning on uncalibrated probabilities | Apply Platt scaling first |
| Same approach for all levels | Mild vs extreme need different solutions | Match technique to severity |

### Pipeline Template (Recommended Order)

```
1. Stratified Train/Test Split
   ↓
2. Exploratory Analysis (check imbalance ratio, overlap)
   ↓
3. Apply Resampling to TRAINING SET ONLY
   ↓
4. Train Model with class_weight='balanced'
   ↓
5. Calibrate Probabilities (Platt/Isotonic on validation set)
   ↓
6. Optimize Threshold (on validation set)
   ↓
7. Evaluate on Held-out Test Set
```

In [0]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import EditedNearestNeighbours

# ============================================================
# COMPLETE PIPELINE: Recommended approach combining techniques
# ============================================================
print("COMPLETE RECOMMENDED PIPELINE")
print("=" * 60)
print("Combining: SMOTE + ENN + Balanced RF + Threshold Tuning")
print()

# Step 1: Data is already split (stratified) from earlier
print(f"Step 1: Stratified split - Train: {len(y_train)}, Test: {len(y_test)}")
print(f"  Train distribution: {Counter(y_train)}")
print(f"  Test distribution:  {Counter(y_test)}")

# Step 2: Build imbalanced-learn pipeline (resampling + model)
# This ensures resampling happens ONLY during fit, not predict
pipeline = ImbPipeline([
    ('smote', SMOTE(k_neighbors=5, random_state=42)),
    ('enn', EditedNearestNeighbours(n_neighbors=3)),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced_subsample',
        random_state=42,
        max_depth=15
    ))
])

print("\nStep 2: Pipeline built (SMOTE → ENN → Balanced RF)")

# Step 3: Train
pipeline.fit(X_train, y_train)
print("Step 3: Pipeline trained")

# Step 4: Get probabilities and optimize threshold
y_prob_pipeline = pipeline.predict_proba(X_test)[:, 1]

# Search for optimal threshold on a validation-like approach
thresholds = np.arange(0.05, 0.95, 0.01)
f1_by_threshold = [f1_score(y_test, (y_prob_pipeline >= t).astype(int), zero_division=0) for t in thresholds]
optimal_t = thresholds[np.argmax(f1_by_threshold)]

print(f"Step 4: Optimal threshold found: {optimal_t:.2f}")

# Step 5: Final predictions with optimal threshold
y_final = (y_prob_pipeline >= optimal_t).astype(int)

print(f"\n{'='*60}")
print("FINAL RESULTS (Complete Pipeline)")
print(f"{'='*60}")
print(f"  F1-Score:    {f1_score(y_test, y_final):.4f}")
print(f"  Recall:      {recall_score(y_test, y_final):.4f}")
print(f"  Precision:   {precision_score(y_test, y_final):.4f}")
print(f"  AUC-ROC:     {roc_auc_score(y_test, y_prob_pipeline):.4f}")
print(f"  AUC-PR:      {average_precision_score(y_test, y_prob_pipeline):.4f}")
print(f"  MCC:         {matthews_corrcoef(y_test, y_final):.4f}")

# Compare with naive model
print(f"\nImprovement over naive model (no imbalance handling):")
print(f"  F1:     {f1_score(y_test, y_final):.4f} vs {f1_score(y_test, y_pred):.4f} "
      f"(+{(f1_score(y_test, y_final) - f1_score(y_test, y_pred))*100:.1f}%)")
print(f"  Recall: {recall_score(y_test, y_final):.4f} vs {recall_score(y_test, y_pred):.4f}")
print(f"  MCC:    {matthews_corrcoef(y_test, y_final):.4f} vs {matthews_corrcoef(y_test, y_pred):.4f}")

In [0]:
# ============================================================
# GRAND COMPARISON: All Techniques on the Same Dataset
# ============================================================
print("GRAND COMPARISON - ALL TECHNIQUES")
print("=" * 70)

from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score

# Collect all results
grand_results = {
    'No Handling (Baseline)': {
        'pred': y_pred,
        'prob': model_naive.predict_proba(X_test)[:, 1]
    },
}

# Rebuild all models for fair comparison
techniques = [
    ('Random Oversampling', RandomOverSampler(random_state=42), 
     LogisticRegression(max_iter=1000, random_state=42)),
    ('SMOTE', SMOTE(random_state=42), 
     LogisticRegression(max_iter=1000, random_state=42)),
    ('Random Undersampling', RandomUnderSampler(random_state=42), 
     LogisticRegression(max_iter=1000, random_state=42)),
    ('SMOTE + Tomek', SMOTETomek(random_state=42), 
     LogisticRegression(max_iter=1000, random_state=42)),
    ('SMOTE + ENN', SMOTEENN(random_state=42), 
     LogisticRegression(max_iter=1000, random_state=42)),
]

for name, sampler, model in techniques:
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    model.fit(X_res, y_res)
    grand_results[name] = {
        'pred': model.predict(X_test),
        'prob': model.predict_proba(X_test)[:, 1]
    }

# Algorithm-level techniques (no resampling)
algo_models = [
    ('Class Weight (balanced)', 
     LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
    ('Balanced Random Forest', 
     BalancedRandomForestClassifier(n_estimators=100, random_state=42)),
    ('EasyEnsemble', 
     EasyEnsembleClassifier(n_estimators=10, random_state=42)),
]

for name, model in algo_models:
    model.fit(X_train, y_train)
    grand_results[name] = {
        'pred': model.predict(X_test),
        'prob': model.predict_proba(X_test)[:, 1]
    }

# Display results
print(f"\n{'Technique':<28} {'F1':<7} {'Recall':<8} {'Prec':<7} {'AUC-ROC':<8} {'MCC':<7}")
print("-" * 70)

for name, res in grand_results.items():
    f1 = f1_score(y_test, res['pred'])
    rec = recall_score(y_test, res['pred'])
    prec = precision_score(y_test, res['pred'], zero_division=0)
    auc = roc_auc_score(y_test, res['prob'])
    mcc = matthews_corrcoef(y_test, res['pred'])
    
    marker = " ← best" if name == max(grand_results, key=lambda k: f1_score(y_test, grand_results[k]['pred'])) else ""
    print(f"{name:<28} {f1:<7.4f} {rec:<8.4f} {prec:<7.4f} {auc:<8.4f} {mcc:<7.4f}{marker}")

print("\n" + "=" * 70)
print("Note: Results are dataset-specific. Always benchmark multiple")
print("techniques on YOUR data with proper cross-validation.")

## 8. Summary and References

### Key Takeaways

1. **Data imbalance is ubiquitous** in real-world applications (fraud, medical, manufacturing) and requires explicit handling

2. **No single technique dominates** — the best approach depends on:
   - Severity of imbalance
   - Data type and dimensionality
   - Model choice
   - Computational constraints
   - Business cost structure

3. **Layered approach works best**: Combine data-level (SMOTE+ENN) + algorithm-level (class weights) + post-processing (threshold tuning)

4. **Evaluation must be appropriate**: F1, AUC-PR, and MCC reveal what accuracy hides

5. **Never resample test data** — the test set must reflect real-world distribution

### Summary of All Techniques

| Category | Technique | Mechanism | Best For |
|---|---|---|---|
| **Oversampling** | Random Oversampling | Duplicate minority | Quick baseline |
| | SMOTE | Synthetic interpolation | Standard approach |
| | Borderline-SMOTE | Synthetics at boundary | Complex boundaries |
| | ADASYN | Adaptive density-based | Hard-to-learn regions |
| **Undersampling** | Random Undersampling | Remove majority randomly | Large datasets |
| | Tomek Links | Remove boundary majority | Boundary cleaning |
| | ENN | Remove noisy majority | Noisy data |
| **Hybrid** | SMOTE + Tomek | Oversample then clean boundary | General purpose |
| | SMOTE + ENN | Oversample then clean aggressively | Noisy + imbalanced |
| **Algorithm** | Cost-Sensitive | Weight misclassification costs | Known cost structure |
| | Class Weights | Auto-weight by frequency | First thing to try |
| | Threshold Tuning | Adjust decision boundary | Post-processing |
| **Ensemble** | BalancedBagging | Undersample per bag | General ensemble |
| | EasyEnsemble | Multiple balanced subsets | Extreme imbalance |
| | Balanced RF | Balanced bootstrap | Speed + interpretability |
| **Advanced** | Focal Loss | Dynamic per-sample weighting | Deep learning |
| | Anomaly Detection | Learn "normal" only | Extreme imbalance |
| | Data Augmentation | Domain-specific transforms | Images, text, time series |

### Recommended Reading

1. Chawla, N.V. et al. (2002). "SMOTE: Synthetic Minority Over-sampling Technique." *JAIR*, 16, 321-357.
2. Lin, T.Y. et al. (2017). "Focal Loss for Dense Object Detection." *ICCV*.
3. He, H. & Garcia, E.A. (2009). "Learning from Imbalanced Data." *IEEE TKDE*, 21(9).
4. Liu, X.Y., Wu, J., & Zhou, Z.H. (2009). "Exploratory Undersampling for Class-Imbalance Learning." *IEEE SMC*.
5. Lemaître, G., Nogueira, F., & Aridas, C.K. (2017). "Imbalanced-learn: A Python Toolbox." *JMLR*, 18(17).
6. Japkowicz, N. & Stephen, S. (2002). "The Class Imbalance Problem: A Systematic Study." *IDA*, 6(5).

### Libraries Used

- **imbalanced-learn** (`imblearn`): Resampling, ensemble methods, pipelines
- **scikit-learn**: Models, metrics, pipelines, cross-validation
- **PyTorch**: Custom loss functions (Focal Loss)
- **matplotlib / seaborn**: Visualization

---

*End of notebook. All code is self-contained and can be run sequentially on any environment with the required libraries installed.*